In [0]:
from pyspark.sql import functions as F

df = spark.read.table("workspace.tfl.bronze_journeys")
df.printSchema()

In [0]:
df.show(20)

In [0]:
# convert data types

def parse_tfl_time(column_name):
    return F.coalesce(
        # attempts to parse string into date, otherwise returns null if none pass
        F.try_to_timestamp(F.col(column_name), F.lit("dd/MM/yyyy HH:mm")),
        F.try_to_timestamp(F.col(column_name), F.lit("yyyy-MM-dd HH:mm"))
    )

# replace end_date with timestamps
typed = df.select(
    F.col("rental_id").cast("bigint"),
    (F.col("duration").cast("int") / 60).alias("duration_minutes"),
    F.col("bike_id").cast("int"),
    parse_tfl_time("start_date").alias("start_time"),
    F.col("start_station_id").cast("int"),
    F.col("start_station_name"),
    parse_tfl_time("end_date").alias("end_time"),
    F.col("end_station_id").cast("int"),
    F.col("end_station_name"),
)

typed.show(10)
typed.printSchema()

In [0]:
typed.count()

In [0]:
# drop duplicates with the same rental_id
no_duplicates = (typed
    .dropDuplicates(["rental_id"]))

print("Removed: ", typed.count() - no_duplicates.count())

In [0]:
# filter out hires with no end station
no_end = (no_duplicates
            .filter(F.col("end_station_id").isNotNull()))
print("Removed: ", no_duplicates.count() - no_end.count())

In [0]:
# filter out hires with duration of 0 minutes
no_zero = no_end.filter(F.col("duration_minutes") > 0)
print("Removed: ", no_end.count() - no_zero.count())


In [0]:
# filter out hires with duration longer than 24 hours
silver = (no_zero
    .filter(F.col("duration_minutes") < (24 * 60))
)
print("Removed: ", no_zero.count() - silver.count())

In [0]:
silver.write.mode("overwrite").saveAsTable("workspace.tfl.silver_journeys")
print(silver.count())
display(silver)